# Descarga y Extracción de Datos: Lending Club Loan Data

Este notebook se encarga de descargar y preparar la base de datos oficial de **Lending Club** utilizada para el modelado de riesgo de crédito (Clasificación de Incumplimiento y Regresión de LGD).

### ¿Por qué fallaba la descarga anteriormente?
1. **Identificador erróneo en Kaggle:** El código anterior intentaba descargar `laotse/credit-risk-dataset`, el cual es un dataset pequeño (12 columnas, 32k registros) y diferente al solicitado.
2. **El dataset oficial:** El dataset correcto en Kaggle es [`wendykan/lending-club-loan-data`](https://www.kaggle.com/datasets/wendykan/lending-club-loan-data), el cual contiene **887.379 créditos y 74 variables**, incluyendo los estados de crédito (`loan_status`) y las variables financieras de recuperación (`total_rec_prncp`, `recoveries`, `collection_recovery_fee`, etc.).
3. **Uso incorrecto de `zipfile`:** `kagglehub.dataset_download()` ya descomprime los archivos automáticamente en el directorio de caché de Kaggle; no es necesario llamar a `zipfile.ZipFile` sobre la ruta retornada.

In [1]:
# Instalación de librerías requeridas (descomentar si es necesario):
# !pip install kagglehub pandas openpyxl

In [5]:
from pathlib import Path
import shutil
import kagglehub
import pandas as pd

# 1. Definir rutas del proyecto
base_dir = Path.cwd()
local_loan_csv = base_dir / "loan" / "loan.csv"
local_dict_xlsx = base_dir / "LCDataDictionary.xlsx"

print(f"Directorio de trabajo: {base_dir}")

# 2. Verificar si los datos ya existen localmente para evitar re-descargas pesadas (~106 MB / 441 MB)
if local_loan_csv.exists():
    print(f"El archivo 'loan.csv' ya existe localmente en: {local_loan_csv}")
else:
    print("El archivo no se encuentra localmente. Iniciando descarga desde Kaggle...")
    try:
        # Identificador oficial de Lending Club en Kaggle
        dataset_handle = "wendykan/lending-club-loan-data"
        
        # Descargar dataset (kagglehub lo descarga y descomprime en la caché local)
        downloaded_dir = Path(kagglehub.dataset_download(dataset_handle))
        print(f"Descarga completada en caché: {downloaded_dir}")
        
        # Copiar archivos relevantes al directorio de trabajo local
        (base_dir / "loan").mkdir(exist_ok=True)
        for file in downloaded_dir.glob("*"):
            if file.suffix.lower() in [".csv", ".xlsx"]:
                dest = base_dir / "loan" / file.name if file.suffix.lower() == ".csv" else base_dir / file.name
                shutil.copy2(file, dest)
                print(f"Copiado {file.name} -> {dest}")
                
    except Exception as e:
        print("\n--- AVISO DE AUTENTICACIÓN KAGGLE ---")
        print("Si la descarga falló por un error 403 (permisos), Kaggle requiere que te autentiques.")
        print("Puedes iniciar sesión ejecutando en una celda:")
        print("    import kagglehub; kagglehub.login()")
        print("O asegurándote de colocar tu 'kaggle.json' en ~/.kaggle/kaggle.json")
        print(f"Detalle del error: {e}")

Directorio de trabajo: c:\Users\ADMON\Documents\Universidad\Redes Neuronales\Trabajos\trabajo_2
El archivo 'loan.csv' ya existe localmente en: c:\Users\ADMON\Documents\Universidad\Redes Neuronales\Trabajos\trabajo_2\loan\loan.csv


In [6]:
# 3. Validación y exploración inicial del dataset
target_csv = local_loan_csv if local_loan_csv.exists() else (base_dir / "loan.csv")

if target_csv.exists():
    print(f"Leyendo información de: {target_csv} (esto puede tomar unos segundos)...\n")
    # Leemos solo algunas columnas clave para inspección rápida
    cols_preview = ['id', 'loan_status', 'loan_amnt', 'funded_amnt', 'total_rec_prncp', 'recoveries']
    df_preview = pd.read_csv(target_csv, usecols=cols_preview, nrows=10000)
    
    print("Muestra de los primeros registros cargados correctamente:")
    display(df_preview.head())
    
    # Verificar las categorías reales de loan_status
    df_status = pd.read_csv(target_csv, usecols=['loan_status'])
    print(f"\nTotal de registros en el dataset: {len(df_status):,}")
    print("\nDistribución de 'loan_status':")
    print(df_status['loan_status'].value_counts(dropna=False))
else:
    print(f"No se encontró el archivo en {target_csv}. Por favor verifica la descarga.")

Leyendo información de: c:\Users\ADMON\Documents\Universidad\Redes Neuronales\Trabajos\trabajo_2\loan\loan.csv (esto puede tomar unos segundos)...

Muestra de los primeros registros cargados correctamente:


,id,loan_amnt,funded_amnt,loan_status,total_rec_prncp,recoveries
0,1077501,5000.0,5000.0,Fully Paid,5000.00,0.00
1,1077430,2500.0,2500.0,Charged Off,456.46,117.08
2,1077175,2400.0,2400.0,Fully Paid,2400.00,0.00
3,1076863,10000.0,10000.0,Fully Paid,10000.00,0.00
4,1075358,3000.0,3000.0,Current,2233.10,0.00



Total de registros en el dataset: 887,379

Distribución de 'loan_status':
loan_status
Current                                                601779
Fully Paid                                             207723
Charged Off                                             45248
Late (31-120 days)                                      11591
Issued                                                   8460
In Grace Period                                          6253
Late (16-30 days)                                        2357
Does not meet the credit policy. Status:Fully Paid       1988
Default                                                  1219
Does not meet the credit policy. Status:Charged Off       761
Name: count, dtype: int64


In [4]:
# 4. Inspeccionar el diccionario de variables (LCDataDictionary.xlsx)
if local_dict_xlsx.exists():
    df_dict = pd.read_excel(local_dict_xlsx)
    print(f"Diccionario de datos cargado ({len(df_dict)} variables descritas):")
    display(df_dict.head(10))
else:
    print(f"No se encontró el diccionario de datos en {local_dict_xlsx}")

Diccionario de datos cargado (80 variables descritas):


c:\Users\ADMON\AppData\Local\Programs\Python\Python311\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


,LoanStatNew,Description
0,addr_state,The state provided by the borrower in the loan...
1,annual_inc,The self-reported annual income provided by th...
2,annual_inc_joint,The combined self-reported annual income provi...
3,application_type,Indicates whether the loan is an individual ap...
4,collection_recovery_fee,post charge off collection fee
5,collections_12_mths_ex_med,Number of collections in 12 months excluding m...
6,delinq_2yrs,The number of 30+ days past-due incidences of ...
7,desc,Loan description provided by the borrower
8,dti,A ratio calculated using the borrower’s total ...
9,dti_joint,A ratio calculated using the co-borrowers' tot...
